<a href="https://colab.research.google.com/github/Harshithpalan/Python-projects/blob/main/Anomaly_Detection_in_Images.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Data Augmentations for Contrastive Learning
In SimCLR, the quality of learned representations depends heavily on strong data augmentations (RandomResizedCrop and ColorJitter are critical).

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, models
from torch.utils.data import DataLoader, Dataset

class ContrastiveTransformations:
    def __init__(self, base_transforms, n_views=2):
        self.base_transforms = base_transforms
        self.n_views = n_views

    def __call__(self, x):
        return [self.base_transforms(x) for i in range(self.n_views)]

# Define augmentations
contrast_transforms = transforms.Compose([
    transforms.RandomResizedCrop(size=32),
    transforms.RandomHorizontalFlip(),
    transforms.RandomApply([transforms.ColorJitter(0.8, 0.8, 0.8, 0.2)], p=0.8),
    transforms.RandomGrayscale(p=0.2),
    transforms.GaussianBlur(kernel_size=3),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

### 2. The SimCLR Model Architecture
We use a base encoder (ResNet) and a projection head (MLP) to map features into a space where contrastive loss is applied.

In [2]:
class SimCLR(nn.Module):
    def __init__(self, base_model, out_dim=128):
        super(SimCLR, self).__init__()
        self.encoder = models.resnet18(weights=None)
        self.encoder.fc = nn.Identity()  # Remove final layer

        # Projection head
        self.projection_head = nn.Sequential(
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, out_dim)
        )

    def forward(self, x):
        h = self.encoder(x)
        z = self.projection_head(h)
        return z

### 3. Contrastive Loss (NT-Xent)
This loss function encourages the model to maximize agreement between two views of the same image.

In [3]:
def info_nce_loss(features, batch_size, temperature=0.5):
    labels = torch.cat([torch.arange(batch_size) for i in range(2)], dim=0)
    labels = (labels.unsqueeze(0) == labels.unsqueeze(1)).float()
    labels = labels.to(features.device)

    features = F.normalize(features, dim=1)

    similarity_matrix = torch.matmul(features, features.T)

    # Mask out self-similarities
    mask = torch.eye(labels.shape[0], dtype=torch.bool).to(features.device)
    labels = labels[~mask].view(labels.shape[0], -1)
    similarity_matrix = similarity_matrix[~mask].view(similarity_matrix.shape[0], -1)

    # Select positives
    positives = similarity_matrix[labels.bool()].view(labels.shape[0], -1)

    # Select negatives
    negatives = similarity_matrix[~labels.bool()].view(similarity_matrix.shape[0], -1)

    logits = torch.cat([positives, negatives], dim=1)
    labels = torch.zeros(logits.shape[0], dtype=torch.long).to(features.device)

    logits = logits / temperature
    return F.cross_entropy(logits, labels)

### 4. Training Loop and Data Loading
We use the STL-10 dataset (unlabeled split) and initialize the SimCLR model with a ResNet-18 backbone. This allows the model to learn features from a large pool of unlabeled images.

In [ ]:
from torchvision.datasets import STL10

# Load dataset with dual-view transformations
train_dataset = STL10(root='./data', split='unlabeled', download=True,
                      transform=ContrastiveTransformations(contrast_transforms, n_views=2))

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, drop_last=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimCLR(base_model="resnet18").to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-4)

def train(epochs=1):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for (images, _) in train_loader:
            # images is a list of [view1, view2]
            images = torch.cat(images, dim=0).to(device)

            optimizer.zero_grad()
            features = model(images)
            # batch_size is divided by 2 because images contains two views per sample
            loss = info_nce_loss(features, batch_size=images.shape[0]//2)

            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

# Execute a single training epoch as a demonstration
train(epochs=1)

### 4. Training Loop and Data Loading
We use the STL-10 dataset (unlabeled split) and initialize the SimCLR model with a ResNet-18 backbone.

In [ ]:
from torchvision.datasets import STL10

# Load dataset with dual-view transformations
train_dataset = STL10(root='./data', split='unlabeled', download=True,
                      transform=ContrastiveTransformations(contrast_transforms, n_views=2))

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, drop_last=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimCLR(base_model="resnet18").to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-4)

def train(epochs=1):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for (images, _) in train_loader:
            # images is a list of [view1, view2]
            images = torch.cat(images, dim=0).to(device)

            optimizer.zero_grad()
            features = model(images)
            loss = info_nce_loss(features, batch_size=128)

            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

# Execute a single training epoch as a demonstration
train(epochs=1)

### 4. Training Loop Setup
We will use the STL-10 dataset, which is designed for unsupervised learning, to train our model.

In [ ]:
from torchvision.datasets import STL10

# Load dataset with dual-view transformations
train_dataset = STL10(root='./data', split='unlabeled', download=True,
                      transform=ContrastiveTransformations(contrast_transforms, n_views=2))

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, drop_last=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimCLR(base_model="resnet18").to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-4)

def train(epochs=1):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for (images, _) in train_loader:
            # images is a list of two views
            images = torch.cat(images, dim=0).to(device)

            optimizer.zero_grad()
            features = model(images)
            loss = info_nce_loss(features, batch_size=128)

            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

# Start a short training run
train(epochs=1)

100%|██████████| 2.64G/2.64G [01:04<00:00, 41.1MB/s]
